In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 1) Load Quantiacs data + pick random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()  # DataArray(field, time, asset)
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns matrix R (daily) from close
# ============================================================
def compute_returns_matrix(data, chosen_assets, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")  # DataArray(time, asset)
    times = rets.time.values
    R = rets.values  # (T, n)
    return times, R


# ============================================================
# 3) Rolling Sigma_t (covariance) - annualized
# ============================================================
def compute_sigma_series(R, lookback=252, annualize=252):
    T, n = R.shape
    if T <= lookback + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback+1}.")

    Sigma_series = np.zeros((T, n, n))
    for t in range(T):
        start = max(0, t - lookback)
        window = R[start:t+1, :]  # inclusive end at t

        if window.shape[0] > 2:
            Sigma = np.cov(window, rowvar=False)
        else:
            Sigma = np.eye(n) * 1e-6

        Sigma_series[t] = Sigma * annualize

    return Sigma_series


# ============================================================
# 4) Switch (b): build mu_series in two different ways
# ============================================================
def mu_series_simple(R, lookback_mu=60, mode="momentum", annualize=252):
    """
    Simple predictor:
      momentum: mean of last K returns
      mean_reversion: negative mean of last K returns
    Uses only data up to time t (inclusive).
    """
    T, n = R.shape
    mu = np.zeros((T, n))
    for t in range(T):
        start = max(0, t - lookback_mu + 1)
        window = R[start:t+1, :]
        m = np.nanmean(window, axis=0)
        if mode == "mean_reversion":
            m = -m
        mu[t] = m * annualize
    return mu


def mu_series_ar1(R, lookback_model=252, annualize=252, smooth_alpha=0.20):
    """
    Rolling AR(1) per asset via OLS:
        r_tau = a + b * r_{tau-1} + eps
    Forecast at time t:
        mu_t = a_hat + b_hat * r_t

    Optional smoothing (EMA):
        mu_sm[t] = alpha * mu[t] + (1-alpha) * mu_sm[t-1]
    This reduces noisy sign-flips and huge turnover.
    """
    T, n = R.shape
    mu_raw = np.zeros((T, n))

    for t in range(T):
        start = max(1, t - lookback_model + 1)
        window = R[start:t+1, :]
        if window.shape[0] < 3:
            mu_raw[t] = 0.0
            continue

        X = window[:-1, :]
        Y = window[1:, :]

        for i in range(n):
            x = X[:, i]
            y = Y[:, i]
            A = np.column_stack([np.ones_like(x), x])
            theta, *_ = np.linalg.lstsq(A, y, rcond=None)
            a, b = theta
            mu_raw[t, i] = (a + b * R[t, i]) * annualize

    # EMA smoothing across time (helps reduce turnover)
    if smooth_alpha is None or smooth_alpha <= 0.0:
        return mu_raw

    mu_sm = np.zeros_like(mu_raw)
    mu_sm[0] = mu_raw[0]
    for t in range(1, T):
        mu_sm[t] = smooth_alpha * mu_raw[t] + (1.0 - smooth_alpha) * mu_sm[t - 1]
    return mu_sm


# ============================================================
# 5) PSD enforcement (fixes CVXPY DCPError)
# ============================================================
def make_psd(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return cp.psd_wrap(cp.Constant(S))


# ============================================================
# 6) Optimization (one-step / two-step)
# ============================================================
def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []
    n = w.shape[0]
    if fully_invested:
        cons.append(cp.sum(w) == 1)
    if long_only:
        cons.append(w >= 0)
    if w_min is not None:
        cons.append(w >= w_min * np.ones(n))
    if w_max is not None:
        cons.append(w <= w_max * np.ones(n))
    return cons


def solve_one_step(mu, Sigma, w_prev, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    n = len(mu)
    w = cp.Variable(n)
    u = w - w_prev
    TC = cp.norm1(u)

    Sigma_psd = make_psd(Sigma, ridge=ridge)

    obj = cp.Maximize(mu @ w - gamma * cp.quad_form(w, Sigma_psd) - lam * TC)
    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max))
    prob.solve(solver=cp.OSQP, warm_start=True, verbose=False)
    return w.value


def run_one_step(mu_series, Sigma_series, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n
    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step(mu_series[t], Sigma_series[t], w_prev,
                             gamma=gamma, lam=lam, w_max=w_max, ridge=ridge)
        W[t] = w_t
        w_prev = w_t
    return W


def turnover_from_weights(W):
    # turnover from rebalancing t-1 -> t
    return np.sum(np.abs(W[1:] - W[:-1]), axis=1)


def equity_from_weights_and_returns(R_daily, W, cost_rate=0.0005):
    """
    R_daily[t] is return from t-1 -> t.
    Use W[t-1] for R_daily[t] (causal).
    Transaction costs charged on turnover from W[t-1] -> W[t].
    """
    rp_gross = np.sum(R_daily[1:] * W[:-1], axis=1)  # length T-1
    to = turnover_from_weights(W)                    # length T-1
    rp_net = rp_gross - cost_rate * to
    eq_gross = np.cumprod(1 + rp_gross)
    eq_net = np.cumprod(1 + rp_net)
    return eq_gross, eq_net, rp_gross, rp_net, to


# ============================================================
# 7) Run: Simple vs AR(1) (gross + net)
# ============================================================
if __name__ == "__main__":
    # --- settings ---
    seed = 7
    n_assets = 50
    lookback_sigma = 252
    K_simple = 60
    ar_window = 252
    annualize = 252

    gamma = 5.0
    lam_simple = 0.06          # turnover penalty for momentum
    lam_ar1 = 0.20             # stronger turnover penalty for noisy AR(1)
    w_max = 0.10
    ridge = 1e-3

    cost_rate = 0.0005         # 5 bps per 100% turnover (try 0.0002..0.001)
    smooth_alpha = 0.20        # AR(1) forecast smoothing (0=no smoothing)

    # --- load data ---
    data, chosen_assets = load_quantiacs_universe(seed=seed, n_assets=n_assets)
    print("Chosen assets (first 10):", chosen_assets[:10], " ... total:", len(chosen_assets))

    times, R_daily = compute_returns_matrix(data, chosen_assets, use_log_returns=False)

    # --- risk model ---
    Sigma_series = compute_sigma_series(R_daily, lookback=lookback_sigma, annualize=annualize)

    # --- alphas ---
    mu_simple = mu_series_simple(R_daily, lookback_mu=K_simple, mode="momentum", annualize=annualize)
    mu_ar1 = mu_series_ar1(R_daily, lookback_model=ar_window, annualize=annualize, smooth_alpha=smooth_alpha)

    # --- optimize (one-step for both) ---
    W_simple = run_one_step(mu_simple, Sigma_series, gamma=gamma, lam=lam_simple, w_max=w_max, ridge=ridge)
    W_ar1    = run_one_step(mu_ar1,    Sigma_series, gamma=gamma, lam=lam_ar1,    w_max=w_max, ridge=ridge)

    # --- sanity prints ---
    print("Any NaNs in W_simple?", np.isnan(W_simple).any())
    print("Any NaNs in W_ar1?", np.isnan(W_ar1).any())
    print("Avg max weight (simple):", np.mean(W_simple.max(axis=1)))
    print("Avg max weight (ar1):", np.mean(W_ar1.max(axis=1)))
    print("Avg turnover (simple):", np.mean(turnover_from_weights(W_simple)))
    print("Avg turnover (ar1):", np.mean(turnover_from_weights(W_ar1)))

    # --- equity (gross + net) ---
    eq_simple_g, eq_simple_n, _, _, to_simple = equity_from_weights_and_returns(R_daily, W_simple, cost_rate=cost_rate)
    eq_ar1_g,    eq_ar1_n,    _, _, to_ar1    = equity_from_weights_and_returns(R_daily, W_ar1,    cost_rate=cost_rate)

    # ============================================================
    # 8) Plots
    # ============================================================
    plt.figure()
    plt.plot(eq_simple_g, label="Momentum (gross)")
    plt.plot(eq_ar1_g, label="AR(1) (gross)")
    plt.title("Gross equity (weights shifted by 1 day)")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure()
    plt.plot(eq_simple_n, label="Momentum (net)")
    plt.plot(eq_ar1_n, label="AR(1) (net)")
    plt.title(f"Net equity with transaction costs (cost_rate={cost_rate:.4f})")
    plt.xlabel("t")
    plt.ylabel("equity")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure()
    plt.plot(to_simple, label="Momentum turnover")
    plt.plot(to_ar1, label="AR(1) turnover")
    plt.title("Turnover per rebalance: sum |Δw|")
    plt.xlabel("t")
    plt.ylabel("turnover")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
print("Any NaNs in W_simple?", np.isnan(W_simple).any())
print("Any NaNs in W_ar1?", np.isnan(W_ar1).any())

print("Avg max weight (simple):", np.mean(W_simple.max(axis=1)))
print("Avg max weight (ar1):", np.mean(W_ar1.max(axis=1)))

print("Avg turnover (simple):", np.mean(np.sum(np.abs(np.diff(W_simple,axis=0)),axis=1)))
print("Avg turnover (ar1):", np.mean(np.sum(np.abs(np.diff(W_ar1,axis=0)),axis=1)))